# `expansion_hunter_03_setup` — create or change a cohort run

Run **once per cohort** in a VWB JupyterLab app, then use
[`expansion_hunter_05_submit.ipynb`](expansion_hunter_05_submit.ipynb) and
[`expansion_hunter_06_progress.ipynb`](expansion_hunter_06_progress.ipynb) without edits.
This notebook never submits Batch jobs.

**Input:** by default this notebook builds the sample table itself with
[`scripts/join_aou2_100k_cram_sex.sh`](../../scripts/join_aou2_100k_cram_sex.sh): the 100k person ids
(`~/aou2_100k.tsv.gz`) joined to DRAGEN sex ploidy and v9 CRAM paths from the workspace's AoU v9 mount.
Sex is `male` for XY and `female` for XX. ExpansionHunter only takes those two, so other ploidies (blank in
the table) are genotyped as female and counted below. Ids without a CRAM are dropped and counted.

To use a different table instead, set `SAMPLES_FILE` to a CSV or TSV (local path or `gs://`) with sample id,
CRAM, CRAI and sex columns.

**Writes** under `batchRuns/expansion_hunter/runs/{RUN_ID}/`: `run.json` (settings), `samples.csv`
(submission order), and the `ACTIVE_RUN` pointer that 05 and 06 read.

**To change a setting later** (for example raise `budget_usd`), edit `SETTINGS` and re-run. The sample list
is left alone unless `REPLACE_SAMPLES = True`, and is never replaced once jobs have been submitted.

Each sample runs minicram then genotype in **one Batch task** on Spot VMs, in its own directory on the VM.

In [ ]:
from __future__ import annotations

import os
import shutil
import sys
from pathlib import Path


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "scripts" / "vwb_batch" / "campaign.py").is_file():
            return p
        nested = p / "aou-lr-phase-2" / "scripts" / "vwb_batch" / "campaign.py"
        if nested.is_file():
            return nested.parents[2]
    raise FileNotFoundError("Cannot find scripts/vwb_batch/campaign.py. git pull kvg/aou-lr-phase-2 and re-run.")


REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT / "scripts"))
for name in [m for m in sys.modules if m == "vwb_batch" or m.startswith("vwb_batch.")]:
    del sys.modules[name]  # pick up a git pull without restarting the kernel

from vwb_batch import alloc, campaign, cohort, cost, gcs, registry  # noqa: E402
from vwb_batch.pipelines import expansion_hunter as eh  # noqa: E402

# ---- edit these ----
RUN_ID = "eh-cohort-v2"
SAMPLES_FILE = ""  # blank: build it below from IDS_FILE with scripts/join_aou2_100k_cram_sex.sh
IDS_FILE = Path.home() / "aou2_100k.tsv.gz"  # person_id list for the join
N_SAMPLES = 0  # 0 = every sample in the table
SHUFFLE_SEED = 1  # submission order is a seeded shuffle, so the canary is a random 200
REPLACE_SAMPLES = False

OUTPUT_BUCKET_GS = "gs://aou-lr-phase2-resources"
CATALOG = f"{OUTPUT_BUCKET_GS}/expansion_hunter/candidate_EH_Loci.GRCh38.degenerate.json"
CATALOG_LOCAL = REPO_ROOT / "expansion_hunter" / "configs" / "candidate_EH_Loci.GRCh38.degenerate.json"
REF_FA = f"{OUTPUT_BUCKET_GS}/locityper/refs/Homo_sapiens_assembly38.fasta"
REF_FAI = REF_FA + ".fai"

SETTINGS = dict(
    out_prefix=f"{OUTPUT_BUCKET_GS}/batchRuns/expansion_hunter/v2",  # per-sample outputs
    budget_usd=3000.0,  # hard cap on estimated spend for this run
    wave_size=10_000,  # most samples in flight at once
    canary_n=200,  # first submission on any new code version
    job_size=1000,  # tasks per Batch job
    spot=True,
    max_attempts=3,  # first attempt + 2 retries; Spot preemptions do not count
    max_failure_rate=0.05,  # halt if more recent first attempts than this fail
    billing_table="",  # project.dataset.table of the detailed billing export, once readable
)
# --------------------

PIPELINE = "expansion_hunter"
PATHS = registry.run_paths(output_bucket=OUTPUT_BUCKET_GS, pipeline=PIPELINE, run_id=RUN_ID)
if shutil.which("gcloud") is None:
    raise SystemExit("gcloud is not on PATH. Open this notebook in a VWB Jupyter app.")
PROJECT, SA = alloc.require_vwb_env()
cfg = campaign.Settings(
    **SETTINGS,
    catalog=CATALOG,
    ref_fa=REF_FA,
    ref_fai=REF_FAI,
    print_reads_image=eh.PRINT_READS_DOCKER,
    eh_image=eh.EH_DOCKER,
)
print("RUN_ID:", RUN_ID, "   ledger:", PATHS.prefix)
print("outputs:", cfg.out_prefix)
print("project:", PROJECT, "   code version:", eh.code_version())

## Reference and catalog

Uploads the 711-locus catalog if it is missing. The reference is shared with Locityper.

In [ ]:
if not gcs.exists(CATALOG):
    print("uploading", CATALOG_LOCAL, "->", CATALOG)
    gcs.upload_file(CATALOG_LOCAL, CATALOG)
for uri in (CATALOG, REF_FA, REF_FAI):
    print("ok " if gcs.exists(uri) else "MISSING ", uri)
if not all(gcs.exists(u) for u in (CATALOG, REF_FA, REF_FAI)):
    raise SystemExit("stage the reference first (expansion_hunter_00_prep_reference)")

## Build the sample table

Runs `scripts/join_aou2_100k_cram_sex.sh` (seconds) unless `SAMPLES_FILE` is set, and keeps a copy of the
table it built in the run folder. Does not submit.

In [ ]:
import subprocess

if not SAMPLES_FILE:
    out = Path.cwd() / "expansion_hunter_rw_scratch" / "aou2_100k_cram_sex.tsv"
    out.parent.mkdir(parents=True, exist_ok=True)
    script = REPO_ROOT / "scripts" / "join_aou2_100k_cram_sex.sh"
    # Blank metrics / manifest arguments use the script's defaults (the workspace's AoU v9 mount).
    proc = subprocess.run(["bash", str(script), str(IDS_FILE), "", "", str(out)], capture_output=True, text=True)
    print(proc.stdout + proc.stderr)
    if proc.returncode != 0:
        raise SystemExit("join failed; see the message above")
    SAMPLES_FILE = str(out)
    gcs.upload_file(out, f"{PATHS.prefix}/samples_source.tsv")
    print("copy:", f"{PATHS.prefix}/samples_source.tsv")
print("SAMPLES_FILE:", SAMPLES_FILE)

## Read and check the sample file

Stops on missing columns. Reports raw sex values, rows dropped, and duplicate ids. Nothing is written yet.

In [ ]:
import random

text = gcs.cat(SAMPLES_FILE) if SAMPLES_FILE.startswith("gs://") else Path(SAMPLES_FILE).read_text(encoding="utf-8")
records, report = cohort.read_sample_table(text)
print("columns used:", report["columns"])
print("samples:", f"{report['n']:,}")
print(f"'{report['columns']['sex']}' column (value: count):")
for raw, n in report["sex_raw"].items():
    flag = "   -> genotyped as female" if raw in report["sex_unrecognized_as_female"] else ""
    print(f"  {n:>9,}  {raw}{flag}")
if report["dropped"]:
    print("dropped rows:", report["dropped"])
if report["duplicates"]:
    print(f"duplicate sample ids skipped: {len(report['duplicates'])} (first: {report['duplicates'][:5]})")

random.Random(SHUFFLE_SEED).shuffle(records)
if N_SAMPLES:
    records = records[:N_SAMPLES]
print("run size:", f"{len(records):,}", "   first canary ids:", [r["sample_id"] for r in records[:5]], "...")

## Spot-check that a few CRAMs are readable

Lists 3 random CRAMs and CRAIs with this workspace as the billing project (the source bucket is requester pays).

In [ ]:
for rec in random.Random(0).sample(records, min(3, len(records))):
    for uri in (rec["cram"], rec["crai"]):
        proc = subprocess.run(
            ["gcloud", "storage", "ls", f"--billing-project={PROJECT}", uri], capture_output=True, text=True
        )
        print("ok " if proc.returncode == 0 else "NOT READABLE ", uri, (proc.stderr or "").strip()[:120])

## Cost preview

Uses the planning cost per sample (Sep 29 on-demand rate plus headroom). The submit notebook switches to this
run's own measured cost once 500 samples have finished, and never commits more than `budget_usd`.

In [ ]:
unit = cfg.prior_unit_usd
print(f"{len(records):,} samples x ${unit:.4f} = ${len(records) * unit:,.0f} planned (budget ${cfg.budget_usd:,.0f})")
spot_vm_share = 0.42 * (1 - cost.SPOT_FRACTION)
print(f"with Spot VMs, expect roughly ${len(records) * 0.023 * (1 - spot_vm_share):,.0f} "
      f"(VMs are ~42% of cost; Spot price assumed {cost.SPOT_FRACTION:.0%} of on-demand)")
if len(records) * unit > cfg.budget_usd:
    print(f"NOTE: the budget covers about {int(cfg.budget_usd / unit):,} samples at the planning rate.")

## Write the run

Writes `run.json`, `samples.csv` (unless they exist and `REPLACE_SAMPLES` is off), and points `ACTIVE_RUN`
at this run.

In [ ]:
state = campaign.load_state(PATHS)
have_samples = gcs.exists(campaign.samples_uri(PATHS))
if have_samples and not REPLACE_SAMPLES:
    print("samples.csv already exists; keeping it (REPLACE_SAMPLES is off)")
elif have_samples and state.get("jobs"):
    raise SystemExit("this run has submitted jobs; start a new RUN_ID instead of replacing its samples")
else:
    campaign.write_samples(PATHS, records)
    print("wrote", campaign.samples_uri(PATHS), f"({len(records):,} samples)")

meta = registry.load_run_json(PATHS)
old = meta.get("campaign") or {}
meta.update({"kind": "campaign", "campaign": cfg.to_dict(), "samples_file": SAMPLES_FILE, "shuffle_seed": SHUFFLE_SEED})
registry.write_run_json(PATHS, meta)
changed = {k: (old.get(k), v) for k, v in cfg.to_dict().items() if old.get(k) != v}
registry.append_event(PATHS, "settings_written", changed=changed)
campaign.set_active_run(OUTPUT_BUCKET_GS, PIPELINE, RUN_ID)
print("wrote", PATHS.run_json)
print("changed:", {k: v[1] for k, v in changed.items()} if old else "(new run)")
print("ACTIVE_RUN ->", RUN_ID)

## Dry run of the canary job

Builds the first job JSON exactly as 05 would and checks its size. Does not submit.

In [ ]:
recs = campaign.load_samples(PATHS, cfg)[: cfg.canary_n]
build_rows, build_job = eh.campaign_builders(cfg, project=PROJECT, sa=SA)
job = build_job(rows=build_rows(recs), tasks_tsv_uri="gs://dry-run/tasks.tsv", labels={"run-id": RUN_ID}, parallelism=len(recs))
group = job["taskGroups"][0]
print("tasks:", group["taskCount"], "   JSON bytes:", f"{campaign.submit.assert_job_json_ok(job):,}")
print("compute:", group["taskSpec"]["computeResource"], "   instances:", job["allocationPolicy"].get("instances"))
print("runnables:", [("container " + r["container"]["imageUri"].rsplit("/", 1)[-1]) if "container" in r else "host script" for r in group["taskSpec"]["runnables"]])
print("Next: open expansion_hunter_05_submit and Run All.")